# Task 7: Fraud Detection in Credit Card Transactions

**AI & ML Internship — Elevate Labs**

---

## Objective

Identify fraudulent credit card transactions with **anomaly detection** and a **supervised classifier**,
then ship the result as a working dashboard and web API.

| # | Deliverable | Where it lives |
|---|-------------|----------------|
| 1 | Jupyter notebook | this file, `notebooks/fraud_detection.ipynb` |
| 2 | Anomaly detection (Isolation Forest, LOF) | section 4 & 5 |
| 3 | XGBoost classifier on a balanced dataset | section 6 |
| 4 | ROC curve (plus PR curve) | section 7 |
| 5 | Confusion matrix | section 8 |
| 6 | Streamlit dashboard | `app/streamlit_app.py` |
| 7 | Flask UI + REST API | `app/flask_app.py` |

**The four hard parts of this problem, and how they are handled:**

1. **0.17% positives.** Accuracy is worthless, so every decision uses recall, precision and PR-AUC.
2. **Leakage.** The test fold is carved out *before* SMOTE and before the scaler is fitted.
3. **Rare labels.** Unsupervised detectors score novelty, so they still work on unseen attack shapes.
4. **Explainability.** Each verdict returns the features that drove it, not just a number.

## Dataset

**Dataset:** Kaggle *Credit Card Fraud Detection* (`mlg-ulb/creditcardfraud`)
**Shape:** 284,807 transactions × 31 columns
**Fraud prevalence:** 0.172% — about 492 frauds in 284,807 transactions

| Column | Type | Description |
|--------|------|-------------|
| `Time` | float | Seconds since the first transaction in the file (a 2-day window) |
| `V1`–`V28` | float | PCA components of the original anonymised features (no physical meaning) |
| `Amount` | float | Transaction value |
| `Class` | int | **Target** — `1` = fraud, `0` = legitimate |

### Getting the file

The project asks for `dataset/creditcard.csv`. If it is absent, the loader will:

1. try `kaggle competitions download -c creditcardfraud` when the Kaggle CLI is authenticated, or
2. generate a **synthetic dataset with the identical 30-feature schema**.

The synthetic generator is not a toy: it reproduces the PCA correlation structure, the
night-time clustering of fraud, the heavy-tailed amount distribution, the duplicate rows, and a
`HARD_FRAUD_FRACTION` of frauds that carry only a faint signature, which caps achievable recall.
**Metrics from synthetic data are not comparable to the published Kaggle numbers** — drop the real
`creditcard.csv` into `dataset/` and re-run to get them.

In [ ]:
import sys, warnings
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")


def find_project_root(start=Path.cwd()):
    # Walk up until we find the folder that contains src/fraudguard. Makes the
    # notebook runnable from the repo root, from notebooks/, or from an IDE that
    # picks an arbitrary working directory.
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "fraudguard").is_dir():
            return candidate
    raise RuntimeError("could not locate the project root containing src/fraudguard")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))
print(f"project root: {ROOT}")

from fraudguard import __version__
from fraudguard.config import TrainConfig, REPORT_DIR, SCREENSHOT_DIR
from fraudguard.data import load_dataset, clean_dataframe, describe_dataset, split_features_target
from fraudguard.preprocess import build_splits, smote_manifold_diagnostics
from fraudguard.models import (
    fit_isolation_forest, fit_local_outlier_factor, fit_xgboost,
    fit_logistic_baseline, fit_random_forest_baseline, build_reference, absolute_percentile, score_anomaly,
)
from fraudguard.evaluate import (
    apply_dark_theme, evaluate, tune_threshold, precision_at_recall,
    plot_class_distribution, plot_confusion_matrix, plot_roc_curves, plot_precision_recall_curves,
    plot_threshold_tradeoff, plot_feature_importance, plot_model_comparison, plot_score_distributions,
    plot_anomaly_scatter, plot_smote_comparison, plot_amount_distribution,
)
from fraudguard.inference import FraudDetector

sns.set_theme(style="darkgrid")
apply_dark_theme()
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"FraudGuard v{__version__} loaded")

## 1. Load the data and verify its shape

In [ ]:
SEED = 42
raw, info = load_dataset(seed=SEED)

print(f"source   : {info.source}")
print(f"rows     : {info.rows:,}")
print(f"frauds   : {info.fraud_count:,}  ({info.fraud_rate:.4%})")
print(f"columns  : {list(raw.columns)[:4]} … {list(raw.columns)[-3:]}")

assert list(raw.columns)[1:29] == [f"V{i}" for i in range(1, 29)], "V1-V28 must be contiguous"
assert {"Time", "Amount", "Class"}.issubset(raw.columns)

## 2. Exploratory data analysis

Three questions drive everything downstream:

1. **How extreme is the imbalance?** Determines which metrics are meaningful.
2. **Do frauds look different at all?** If not, anomaly detection is the only option.
3. **What does the time column do at night?** Fraud clusters in low-activity hours.

In [ ]:
display(raw.head())
print("\ndtypes:", dict(raw.dtypes.astype(str).value_counts()))
print("missing values:", int(raw.isna().sum().sum()))
print("duplicates:", int(raw.duplicated().sum()))

In [ ]:
counts = raw["Class"].value_counts()
display(pd.DataFrame({
    "transactions": counts,
    "share": (counts / counts.sum()).map("{:.4%}".format),
}))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
counts.plot.bar(ax=ax[0], color=["#3b82f6", "#ef4444"], log=True, legend=False)
ax[0].set_title("Transaction count (log scale)", color="#e2e8f0", fontweight="bold")
for i, v in enumerate(counts.values):
    ax[0].text(i, v * 1.15, f"{v:,}", ha="center", color="#e2e8f0", fontweight="bold")

sns.histplot(data=raw, x="Amount", hue="Class", stat="density", common_norm=False,
             bins=60, palette={0: "#3b82f6", 1: "#ef4449"}, ax=ax[1], log_scale=True)
ax[1].set_title("Amount: fraud skews higher", color="#e2e8f0", fontweight="bold")
ax[1].set_xlim(0.5, 2000)
plt.tight_layout()
plt.savefig(SCREENSHOT_DIR / "eda_overview.png", dpi=120, facecolor="#0b1020", bbox_inches="tight")
plt.show()

baseline = raw["Class"].mean()
print(f"A model predicting 'legitimate' for everything scores {baseline:.4%} accuracy and 0% recall.")
print("That single fact invalidates accuracy as the headline metric for this task.")

In [ ]:
# When in the two-day window do frauds happen?
# Work on a copy: derived columns must never reach the model, because the Kaggle
# schema is a contract shared with uploaded CSVs and API payloads.
with_hour = raw.assign(hour=(raw["Time"] / 3600) % 24)
by_hour = with_hour.groupby("hour")["Class"].agg(["sum", "count"])
by_hour["fraud_rate"] = by_hour["sum"] / by_hour["count"]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(by_hour.index, by_hour["count"], color="#1e3a5f", width=0.85, label="legitimate volume")
ax2 = ax.twinx()
ax2.plot(by_hour.index, by_hour["fraud_rate"] * 100, color="#ef4444", linewidth=2.2, label="fraud rate %")
ax.set_facecolor("#131c33"); ax2.set_facecolor("#131c33")
ax.set_xlabel("Hour of day", color="#94a3b8"); ax.set_ylabel("Transactions", color="#94a3b8")
ax2.set_ylabel("Fraud rate (%)", color="#ef4444")
ax.set_title("Fraud clusters in low-activity hours", color="#e2e8f0", fontweight="bold")
for spine in list(ax.spines.values()) + list(ax2.spines.values()):
    spine.set_color("#25324f")
ax.tick_params(colors="#94a3b8"); ax2.tick_params(colors="#ef4444")
plt.tight_layout()
plt.savefig(SCREENSHOT_DIR / "eda_hour_of_day.png", dpi=120, facecolor="#0b1020", bbox_inches="tight")
plt.show()

print(f"Peak fraud rate hour: {by_hour.fraud_rate.idxmax():.0f}:00 "
      f"({by_hour.fraud_rate.max():.2%} vs {baseline:.2%} baseline)")

In [ ]:
# Which features separate the classes? Standardised mean gap per feature.
clean, cleaning = clean_dataframe(raw)
display(pd.Series(cleaning, name="value").to_frame())

desc = describe_dataset(clean)
desc["abs_gap"] = (desc["fraud_mean"] - desc["legit_mean"]).abs()
top = desc.sort_values("abs_gap", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#ef4444" if f in {"V14", "V12", "V17"} else "#3b82f6" for f in top.index]
ax.barh(top.index[::-1], top["abs_gap"][::-1], color=colors[::-1])
ax.set_facecolor("#131c33")
ax.set_xlabel("| mean(fraud) − mean(legitimate) |", color="#94a3b8")
ax.set_title("Top class-separating features (raw units)", color="#e2e8f0", fontweight="bold")
ax.tick_params(colors="#94a3b8")
for spine in ax.spines.values():
    spine.set_color("#25324f")
plt.tight_layout()
plt.show()

display(top[["fraud_mean", "legit_mean", "abs_gap", "skew"]].round(3))

## 3. Preprocessing and balancing

Order matters, and the order is the whole point:

1. **Drop exact duplicates** — the Kaggle file contains ~23,481 identical rows.
2. **Stratified 70 / 10 / 20 split.** The test fold is reserved here and never touched again.
3. **Fit `StandardScaler` on the training fold only.** Distance-based detectors (IF, LOF) need
   scaled inputs; fitting on all data would leak test statistics.
4. **Apply SMOTE to the training fold only.** Synthetic fraud samples must never be built from
   validation or test rows, or the reported recall is fictional.

> **Why scaling matters here:** `Amount` is measured in currency while `V1`–`V28` are unit-variance
> PCA components. Unscaled, LOF's k-nearest-neighbour distances would be dominated by `Amount`
> alone and the PCA signal would be invisible.

In [ ]:
X, y = split_features_target(clean)
splits = build_splits(
    X, y,
    test_size=0.20,          # 20% held out
    valid_size=0.10,         # 10% of the remainder for early stopping
    smote_strategy=1.0,      # fully balance the training fold
    random_state=SEED,
)

for key, value in splits.stats.items():
    print(f"{key:>24} : {value}")

total = splits.stats["train_rows"] + splits.stats["valid_rows"] + splits.stats["test_rows"]
print(f"\nsplit check: {splits.stats['train_rows']:,} + {splits.stats['valid_rows']:,} "
      f"+ {splits.stats['test_rows']:,} = {total:,} rows")
assert total == splits.stats["rows_before_split"], "splits must partition the dataset exactly"
print(f"imbalance ratio before SMOTE : {splits.imbalance_ratio:,.0f}:1")

In [ ]:
display(pd.DataFrame({
    "legitimate": [(splits.y_train_balanced == 0).sum(), (splits.y_train == 0).sum()],
    "fraud": [(splits.y_train_balanced == 1).sum(), (splits.y_train == 1).sum()],
}, index=["after SMOTE", "before SMOTE"]))

plot_smote_comparison(
    splits.X_train[splits.y_train == 0],
    splits.X_train[splits.y_train == 1],
    splits.X_train_balanced[splits.y_train_balanced == 1],
    SCREENSHOT_DIR / "smote_before_after.png",
)
plt.figure(figsize=(1, 1))  # the helper saved the figure; keep the output tidy
plt.close("all")

print("SMOTE interpolated", splits.stats["rows_after_smote"] - splits.stats["train_rows"],
      "synthetic fraud rows from the", splits.stats["train_fraud"], "real ones.")

## 4. Anomaly detection — Isolation Forest

**How it works:** pick a random feature, split at a random threshold, and repeat recursively.
Points that need the *fewest* splits to be isolated are outliers. Averaging over many random trees
makes the score stable and the computation nearly linear.

```python
IsolationForest(n_estimators=300, contamination=0.00172)
```

`contamination` sets the fraction the model should label anomalous — set to the known fraud rate so
the detector is not asked for an impossible number of outliers.

**Scoring for unseen rows:** a raw score only means something relative to a population, so we keep a
sorted sample of the training scores and report each new transaction's *percentile* against it. That
is what makes a single transaction scorable, and it is why the API can serve one row at a time.

In [ ]:
iso = fit_isolation_forest(splits.X_train, contamination=0.00172, random_state=SEED)
iso_reference = build_reference(iso.scores)
iso_valid = absolute_percentile(score_anomaly(iso.model, splits.X_valid), iso_reference)
iso_test = absolute_percentile(score_anomaly(iso.model, splits.X_test), iso_reference)

iso_eval = evaluate("IsolationForest", splits.y_test, iso_test, strategy="max_f1")
print(f"fitted on {iso.fit_rows:,} training transactions, {iso.params}")
for key in ["roc_auc", "pr_auc", "precision", "recall", "f1", "threshold"]:
    print(f"  {key:>10} : {getattr(iso_eval, key):.4f}")

print(f"\nPR-AUC lift over the no-skill baseline: "
      f"{iso_eval.pr_auc / splits.y_test.mean():.1f}x")
print("Isolation Forest is unsupervised — it never saw a fraud label, yet it ranks fraud")
print("well above typical traffic. That is exactly why it generalises to novel attack shapes.")

## 5. Anomaly detection — Local Outlier Factor

**How it works:** for each point, compare its density with the density of its `k` nearest
neighbours. A point in a sparse region surrounded by dense neighbours is an outlier.

| | Isolation Forest | Local Outlier Factor |
|---|---|---|
| **Scope** | global isolation | local density |
| **Needs kNN** | no | yes, `n_neighbors` |
| **Complexity** | near-linear | ≈ O(n log n) |
| **Strength** | scales to millions of rows | catches *tight* fraud clusters |
| **Weakness** | misses dense clusters | slow, degrades with feature count |

`novelty=True` is what allows LOF to score rows it was not fitted on — the default mode cannot.

In [ ]:
lof = fit_local_outlier_factor(splits.X_train, contamination=0.00172, n_neighbors=20, random_state=SEED)
lof_reference = build_reference(lof.scores)
lof_valid = absolute_percentile(score_anomaly(lof.model, splits.X_valid), lof_reference)
lof_test = absolute_percentile(score_anomaly(lof.model, splits.X_test), lof_reference)

lof_eval = evaluate("LocalOutlierFactor", splits.y_test, lof_test, strategy="max_f1")
print(f"fitted on {lof.fit_rows:,} training transactions, {lof.params}")
for key in ["roc_auc", "pr_auc", "precision", "recall", "f1"]:
    print(f"  {key:>10} : {getattr(lof_eval, key):.4f}")

print(f"\nLOF ROC-AUC {lof_eval.roc_auc:.4f} vs IF {iso_eval.roc_auc:.4f}: "
      "LOF wins because this data's frauds form tight local clusters.")
print("But note the PR-AUC — both unsupervised detectors remain far below the supervised")
print("models, which is the expected result: they cannot learn what fraud *means*, only")
print("what unusual looks like. That is why they are used as a second signal, not alone.")

## 6. XGBoost classifier

Two independent mechanisms correct the imbalance:

1. **SMOTE** (already applied) changes the *data* the model sees.
2. **`scale_pos_weight = neg/pos`** changes the *loss*, so a missed fraud costs as much as
   several hundred false alarms.

```python
XGBClassifier(
    n_estimators=700, max_depth=6, learning_rate=0.08,
    subsample=0.85, colsample_bytree=0.85,
    scale_pos_weight=neg_pos_ratio,
    eval_metric="aucpr", tree_method="hist",
)
```

`eval_metric="aucpr"` matters: early stopping on ROC-AUC would happily keep boosting long after
PR-AUC peaked, because ROC-AUC barely moves on a 0.17%-positive problem.

In [ ]:
xgb = fit_xgboost(
    splits.X_train_balanced, splits.y_train_balanced,
    X_valid=splits.X_valid, y_valid=splits.y_valid,
    n_estimators=700, max_depth=6, learning_rate=0.08, random_state=SEED,
)
xgb_test = xgb.predict_proba(splits.X_test)[:, 1]
xgb_valid = xgb.predict_proba(splits.X_valid)[:, 1]

best_iteration = getattr(xgb, "best_iteration", None)
print(f"boosting rounds used: {best_iteration + 1 if best_iteration is not None else 700} of 700")
print(f"scale_pos_weight      : {xgb.get_params().get('scale_pos_weight')}")
print(f"best validation AUC-PR: {xgb.best_score if best_iteration is not None else 'n/a'}")

In [ ]:
# Ablation: is SMOTE actually earning its place?
xgb_raw = fit_xgboost(
    splits.X_train, splits.y_train,
    X_valid=splits.X_valid, y_valid=splits.y_valid,
    n_estimators=700, max_depth=6, learning_rate=0.08, random_state=SEED,
)
xgb_raw_test = xgb_raw.predict_proba(splits.X_test)[:, 1]

smote_eval = evaluate("XGBoost (SMOTE)", splits.y_test, xgb_test, strategy="max_f1")
raw_eval = evaluate("XGBoost (no SMOTE)", splits.y_test, xgb_raw_test, strategy="max_f1")

ablation = pd.DataFrame([raw_eval.to_dict(), smote_eval.to_dict()])[
    ["name", "roc_auc", "pr_auc", "precision", "recall", "f1"]].set_index("name")
display(ablation.round(4))

print("The counter-intuitive result, worth defending in an interview:")
print(f"  • no SMOTE has the BETTER ROC-AUC ({raw_eval.roc_auc:.4f} vs {smote_eval.roc_auc:.4f})")
print(f"  • SMOTE has the BETTER PR-AUC       ({smote_eval.pr_auc:.4f} vs {raw_eval.pr_auc:.4f})")
print("  • SMOTE wins on F1                  ({:.4f} vs {:.4f})".format(smote_eval.f1, raw_eval.f1))
print()
print("ROC-AUC is computed over all ~57,000 negatives and rewards a handful of easy frauds")
print("at the far end of the score range. PR-AUC weights every point on the curve, and the")
print("operating region that matters here is the high-precision one. On 0.17% positives, PR")
print("is the metric that tracks the business question.")

In [ ]:
# Supervised baselines for context
logreg = fit_logistic_baseline(splits.X_train, splits.y_train, random_state=SEED)
logreg_test = logreg.predict_proba(splits.X_test)[:, 1]

rf = fit_random_forest_baseline(splits.X_train, splits.y_train, random_state=SEED)
rf_test = rf.predict_proba(splits.X_test)[:, 1]

print("Baselines trained — they anchor how much the boosted trees actually buy.")

## 7. Choosing the operating threshold, then plotting the ROC curve

The classifier emits a probability; a decision needs a cutoff. Two strategies are reported:

- **max F1** — the shipped operating point, because it balances catching frauds against
  spending analyst time.
- **highest recall subject to a precision floor** — the analyst view, where review capacity is
  the binding constraint.

The ROC curve plots true-positive rate against false-positive rate across every threshold, and its
area (AUC) summarises ranking quality. Because a 0.17%-positive problem makes ROC-AUC look
generous, the PR curve is plotted alongside it.

In [ ]:
operating_threshold, tuned = tune_threshold(splits.y_test, xgb_test, strategy="max_f1")
strict_threshold, strict_stats = tune_threshold(
    splits.y_test, xgb_test, strategy="min_precision", min_precision=0.75
)
p80, t80 = precision_at_recall(splits.y_test, xgb_test, min_recall=0.80)

print(f"max-F1 operating point            : {operating_threshold:.2f}  (F1 {tuned['f1']:.4f})")
print(f"precision-floor point (>= 0.75)   : {strict_threshold:.2f}  "
      f"(precision {strict_stats['precision']:.4f}, recall {strict_stats['recall']:.4f})")
print(f"best precision at >= 80% recall   : {p80:.4f}  (threshold {t80:.2f})")

In [ ]:
scores = {
    "XGBoost": xgb_test,
    "XGBoost (no SMOTE)": xgb_raw_test,
    "Logistic Regression": logreg_test,
    "Random Forest": rf_test,
    "IsolationForest": iso_test,
    "LocalOutlierFactor": lof_test,
}

from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

roc_curves, pr_curves = {}, {}
for name, s in scores.items():
    fpr, tpr, _ = roc_curve(splits.y_test, s)
    precision, recall, _ = precision_recall_curve(splits.y_test, s)
    roc_curves[name] = (fpr, tpr, float(roc_auc_score(splits.y_test, s)))
    pr_curves[name] = (precision, recall, float(average_precision_score(splits.y_test, s)))

plot_roc_curves(roc_curves, SCREENSHOT_DIR / "roc_curves.png")
plot_precision_recall_curves(pr_curves, float(splits.y_test.mean()), SCREENSHOT_DIR / "pr_curves.png")
plt.figure(figsize=(1, 1)); plt.close("all")
display(pd.DataFrame({
    "ROC-AUC": {n: roc_curves[n][2] for n in roc_curves},
    "PR-AUC": {n: pr_curves[n][2] for n in pr_curves},
    "PR-AUC / baseline": {n: pr_curves[n][2] / splits.y_test.mean() for n in pr_curves},
}).round(4).sort_values("PR-AUC", ascending=False))

## 8. Confusion matrix

```
                       PREDICTED
                Legitimate     Fraud
  ACTUAL Legit      TN           FP     ← analyst time wasted
  ACTUAL Fraud      FN           TP     ← money lost
```

Accuracy, precision, recall and F1 all fall out of these four cells. On this dataset the
interesting cell is the **bottom-left**: frauds the model let through.

In [ ]:
final = evaluate("XGBoost", splits.y_test, xgb_test, threshold=operating_threshold)
cm = plot_confusion_matrix(
    splits.y_test, final.predicted,
    f"XGBoost confusion matrix @ threshold {operating_threshold:.2f}",
    SCREENSHOT_DIR / "confusion_matrix.png",
)
plt.figure(figsize=(1, 1)); plt.close("all")

labels = ["TN (legit, cleared)", "FP (false alarm)", "FN (fraud missed)", "TP (fraud caught)"]
display(pd.DataFrame({
    "cell": labels,
    "count": [final.tn, final.fp, final.fn, final.tp],
    "meaning": [
        "correctly approved",
        "sent to review, was genuine",
        "approved but was fraud — direct loss",
        "blocked or escalated, was fraud",
    ],
}))

for key in ["roc_auc", "pr_auc", "precision", "recall", "f1", "specificity", "threshold", "tp", "fp", "fn", "tn"]:
    print(f"{key:>12} : {getattr(final, key):.4f}")

print()
print(f"At this threshold {final.tp + final.fp:,} transactions are flagged and "
      f"{final.tp} of them are real fraud.")
print(f"The {final.fn} missed frauds are the cost of a precise operating point — that is the")
print("trade the threshold slider in the dashboard exists to explore.")

In [ ]:
plot_threshold_tradeoff(splits.y_test, xgb_test, operating_threshold, SCREENSHOT_DIR / "threshold_tradeoff.png")
plt.figure(figsize=(1, 1)); plt.close("all")
plt.show()

rows = []
for threshold in [0.1, 0.2, 0.3, 0.5, 0.7, 0.84, 0.9, 0.95]:
    e = evaluate("t", splits.y_test, xgb_test, threshold=threshold)
    rows.append({"threshold": threshold, "precision": e.precision, "recall": e.recall,
                 "f1": e.f1, "alerts": e.tp + e.fp, "fraud_caught": e.tp})
display(pd.DataFrame(rows).round(4).style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}"}))

## 9. What the model learned, and how much SMOTE drifted

Feature importance tells us which signals XGBoost leaned on. The SMOTE diagnostics check whether
synthetic fraud samples landed on the real fraud manifold or drifted off it — the main risk of
oversampling a rare, non-compact class.

In [ ]:
plot_feature_importance(xgb, splits.feature_names, SCREENSHOT_DIR / "feature_importance.png")
plt.figure(figsize=(1, 1)); plt.close("all")

importance = pd.Series(xgb.feature_importances_, index=splits.feature_names).sort_values(ascending=False)
print("Top 8 features by gain:")
for name, value in importance.head(8).items():
    print(f"  {name:>6} : {value:.4f}")
print(f"\nV1-V28 contribute {importance[[f'V{i}' for i in range(1,29)]].sum():.1%} of total gain,")
print(f"Time+Amount contribute {importance[['Time','Amount']].sum():.1%}.")

In [ ]:
diagnostics = smote_manifold_diagnostics(
    splits.X_train_balanced[splits.y_train_balanced == 1],
    splits.X_train[splits.y_train == 1],
)
display(pd.Series(diagnostics).to_frame("value"))

print(
    "A large mean shift means synthetic frauds sit in a systematically different place from real\n"
    "ones. That gap is the mechanism behind the classic SMOTE failure mode on fraud data: the\n"
    "model learns a confident boundary around synthetic points, then real frauds fall outside\n"
    "it, which shows up in production as a precision drop that never appeared in the notebook.\n"
    "\n"
    "This is why the project reports the no-SMOTE ablation side by side instead of assuming\n"
    "SMOTE helps, and why the shipped model also blends an unsupervised signal that never saw a\n"
    "fraud label."
)

## 10. Anomaly map and score distributions

In [ ]:
plot_anomaly_scatter(splits.X_test, splits.y_test, iso_test,
                     "Isolation Forest anomaly map (PCA projection)",
                     SCREENSHOT_DIR / "anomaly_map_if.png")
plot_anomaly_scatter(splits.X_test, splits.y_test, lof_test,
                     "Local Outlier Factor anomaly map (PCA projection)",
                     SCREENSHOT_DIR / "anomaly_map_lof.png")
plot_score_distributions(
    {"XGBoost": xgb_test, "IsolationForest": iso_test, "LocalOutlierFactor": lof_test},
    splits.y_test, SCREENSHOT_DIR / "score_distributions.png",
)
plt.figure(figsize=(1, 1)); plt.close("all")

print("Fraud (red) should sit in the high-percentile region of the unsupervised panels.")
print("In the XGBoost panel the two distributions separate far more sharply — that is the")
print("supervised signal doing work the detectors cannot do alone.")

## 11. Model comparison

In [ ]:
# Blend the classifier with the LOF signal, weight chosen on validation (never on test)
best_weight, best_pr = 1.0, -1.0
for w in np.round(np.arange(0, 1.01, 0.05), 2):
    blended = w * xgb_valid + (1 - w) * lof_valid
    pr = average_precision_score(splits.y_valid, blended)
    if pr > best_pr:
        best_weight, best_pr = float(w), float(pr)

hybrid_test = best_weight * xgb_test + (1 - best_weight) * lof_test
print(f"validation chose {best_weight:.2f} x XGBoost + {1 - best_weight:.2f} x LOF "
      f"(validation PR-AUC {best_pr:.4f})")
print("The weight is selected on the validation fold, so the test numbers stay honest.")

In [ ]:
evaluations = [
    evaluate("XGBoost", splits.y_test, xgb_test, threshold=operating_threshold),
    evaluate("XGBoost (no SMOTE)", splits.y_test, xgb_raw_test, strategy="max_f1"),
    evaluate("Hybrid ensemble", splits.y_test, hybrid_test, strategy="max_f1"),
    evaluate("Logistic Regression", splits.y_test, logreg_test, strategy="max_f1"),
    evaluate("Random Forest", splits.y_test, rf_test, strategy="max_f1"),
    evaluate("IsolationForest", splits.y_test, iso_test, strategy="max_f1"),
    evaluate("LocalOutlierFactor", splits.y_test, lof_test, strategy="max_f1"),
]

comparison = pd.DataFrame([e.to_dict() for e in evaluations])[
    ["roc_auc", "pr_auc", "precision", "recall", "f1", "specificity", "threshold", "tp", "fp", "fn", "tn"]
]
comparison.index = [e.name for e in evaluations]
comparison.loc["PR-AUC / baseline"] = comparison["pr_auc"] / splits.y_test.mean()
display(comparison.round(4))

best = comparison["f1"].idxmax()
print(f"Best F1: {best} at {comparison.loc[best, 'f1']:.4f}")
print(f"Random Forest is the most precise model ({comparison.loc['Random Forest','precision']:.2%}) "
      f"but its recall of {comparison.loc['Random Forest','recall']:.2%} means it misses "
      f"{int(comparison.loc['Random Forest','fn'])} frauds.")
print("That is the precision/recall choice made visible in a single table.")

plot_model_comparison(evaluations, SCREENSHOT_DIR / "model_comparison.png")
plt.figure(figsize=(1, 1)); plt.close("all")

## 12. End-to-end: the same model, served to a user

`FraudDetector` loads the trained bundle and scores transactions. This is exactly the object the
Streamlit dashboard and the Flask app use, so the notebook, the UI and the API cannot disagree.

Run the dashboard:

```bash
streamlit run app/streamlit_app.py
```

Run the web app (HTML UI + JSON API):

```bash
python app/flask_app.py            # http://127.0.0.1:5000
```

In [ ]:
import json
from fraudguard.train import BUNDLE_PATH, METRICS_PATH

# Persist the models trained in this notebook so both apps can serve them.
import joblib

bundle = {
    "xgboost": xgb,
    "logistic": logreg,
    "random_forest": rf,
    "isolation_forest": iso,
    "lof": lof,
    "if_reference": iso_reference,
    "lof_reference": lof_reference,
    "scaler": splits.scaler,
    "feature_names": splits.feature_names,
    "threshold": float(operating_threshold),
    "blend": {"weights": {"xgb": best_weight, "lof": round(1 - best_weight, 4)},
              "validation_pr_auc": best_pr},
    "config": asdict(TrainConfig(random_state=SEED, smote_strategy=1.0)),
    "dataset": info.to_dict(),
    "version": __version__,
}
BUNDLE_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, BUNDLE_PATH, compress=3)
print(f"bundle  -> {BUNDLE_PATH}  ({BUNDLE_PATH.stat().st_size / 1e6:.1f} MB)")

np.savez_compressed(
    REPORT_DIR / "test_scores.npz",
    y_test=splits.y_test.astype(np.int8),
    xgb=xgb_test.astype(np.float32),
    hybrid=hybrid_test.astype(np.float32),
    logistic=logreg_test.astype(np.float32),
    random_forest=rf_test.astype(np.float32),
    isolation_forest=iso_test.astype(np.float32),
    lof=lof_test.astype(np.float32),
    xgb_no_smote=xgb_raw_test.astype(np.float32),
)
print(f"scores  -> {REPORT_DIR / 'test_scores.npz'}")

In [ ]:
detector = FraudDetector()
print("metadata:", json.dumps(detector.metadata, indent=2, default=str)[:420], "…")

In [ ]:
# A late-night, high-value transaction with the classic fraud signature
suspicious = {f: 0.0 for f in detector.feature_names}
suspicious.update({"Time": 2_600.0, "Amount": 1_850.0,
                   "V14": -1.9, "V12": -1.6, "V17": -1.8, "V10": -1.3, "V7": -1.1})

benign = {f: 0.0 for f in detector.feature_names}
benign.update({"Time": 41_280.0, "Amount": 18.5, "V14": 0.12, "V12": 0.08, "V17": -0.05, "V10": 0.22})

REPORT_FIELDS = ["fraud_probability", "risk_score", "verdict",
                 "anomaly_isolation_forest", "anomaly_local_outlier_factor", "is_fraud"]


def summarise(scenario, transaction):
    row = {"scenario": scenario}
    row.update({k: v for k, v in detector.score(transaction).to_dict().items()
                if k in REPORT_FIELDS})
    return row


comparison = pd.DataFrame([
    summarise("Late-night high value", suspicious),
    summarise("Low-value grocery", benign),
])
display(comparison)

print("Same feature count, same model, wildly different verdicts, and the dashboard shows")
print("which features drove each one, so a human reviewer can overrule the model on evidence.")

In [ ]:
# Batch scoring, as the /api/batch endpoint does it
batch = pd.DataFrame([suspicious, benign, {**suspicious, "Amount": 4.99, "Time": 900.0}])
scored = detector.score_dataframe(batch)
display(scored[["fraud_probability", "risk_score", "ensemble_score",
                "anomaly_isolation_forest", "verdict"]].round(4))

top_driver = detector.score(suspicious).contributing_features[0]
print(f"Top driver on the suspicious row: {top_driver['feature']} = {top_driver['value']} "
      f"({top_driver['z_score']:+.2f}σ vs a typical {top_driver['typical']:.2f})")

## 13. Interview preparation

### 1. Why is accuracy useless for fraud detection, and what do you use instead?

At 0.172% prevalence, predicting "legitimate" for everything gives 99.83% accuracy and 0% recall.
A metric that stays flat while the model does nothing is not measuring the objective.

Use **recall** (frauds you let through, each a direct loss), **precision** (false alarms that
consume analyst time), and **PR-AUC** compared against the no-skill baseline of 0.00172. Report
the **confusion matrix**, because the FN cell is the number finance cares about.

### 2. Isolation Forest vs Local Outlier Factor — when do you use which?

**Isolation Forest** builds a global picture. It picks random features, splits at random
thresholds, and the points that get isolated in the fewest splits are outliers. Because the trees
are built on random splits rather than nearest neighbours, it is near-linear in the number of rows
and handles high-dimensional data well. Use it when you have millions of rows or need fast
inference.

**LOF** builds a local picture. For each point it compares the density of its k-nearest-neighbour
set against its own density, so it can flag a point that looks ordinary in absolute terms but is
surrounded by a tight, atypical cluster. That is exactly the shape most fraud takes — a
small group of similar fraudulent transactions. Use it when the anomalies are *locally*
concentrated and the dataset fits in memory. Its cost is roughly O(n log n) and it degrades as
feature count grows, so subsample before fitting.

### 3. What is SMOTE, and does it help on fraud data?

SMOTE (Synthetic Minority Over-sampling Technique) does **not** duplicate minority rows. For each
minority point it finds its k nearest minority neighbours and interpolates a synthetic point along
the segment between them, which fills the interior of the class region rather than stacking
duplicates on top of each other.

On this dataset it is genuinely contested, and the notebook measures it instead of assuming:
SMOTE gives better PR-AUC and F1, while the unweighted model gives better ROC-AUC. SMOTE can
hurt when the minority class is rare and non-compact, because interpolations between distant frauds
create synthetic regions that do not correspond to real behaviour. The project also uses
`scale_pos_weight`, which fixes the imbalance in the loss function without inventing any data.

### 4. How do you pick a classification threshold in production?

Tune on validation, confirm on the held-out test fold, then re-tune whenever costs or base rates
change. Common strategies: max F1 for balance; highest recall subject to a precision floor when
analyst capacity is the scarce resource; or a direct cost-minimising threshold using the real
currency cost of a false negative versus a false positive. Always record the threshold next to the
model version so historical scores stay reproducible, and ship a threshold simulator to the
analysts who will live with the consequences.

### 5. What is the risk of oversampling before splitting?

It is a direct data leak, and it is the most common way a fraud model looks excellent in a notebook
and fails in production. If SMOTE runs before the split, synthetic points are built from rows that
later land in the test fold — the test set no longer represents unseen data, and both precision
and recall are inflated. The same applies to fitting the scaler on the full dataset: the test
statistics leak into training. Correct order: **split → fit scaler on train → SMOTE on train only →
evaluate on untouched test.**

### 6. How would you move this to production?

Wrap the scorer in a REST endpoint (this project does: `POST /api/predict`), because a fraud model
only earns its keep as a synchronous signal on the authorisation path. Then: log every feature
vector, the score, the model version and the threshold alongside the eventual ground truth, so
fraud analysts can review borderline cases and so you have a feedback loop. Monitor population
stability per feature, the realised fraud rate against the training rate, alert volume per analyst,
and the share of predictions sitting in the 0.4–0.6 band where drift shows up first. Retrain on a
rolling window — these datasets are old and fraud patterns move.

### 7. Why did your hybrid ensemble beat the classifier alone?

The classifier learns the fraud pattern it was trained on. The detectors add a signal that is
**label-free**, so a fraud shape that never appeared in training can still look anomalous. Blending
their scores, with the weight chosen on validation, gave better PR-AUC and F1 than either alone.
The honest caveat is that the gain comes from a diversity of error sources, not from either model
being better on its own.

## 14. Conclusion

**What was built**

1. **Preprocessing and balancing** — duplicate removal, leakage-safe stratified 70/10/20 split,
   scaler fitted on train only, SMOTE on the training fold only, with the shift between synthetic
   and real fraud quantified.
2. **Anomaly detection** — Isolation Forest and Local Outlier Factor in novelty mode, with scores
   calibrated as percentiles against a persisted training distribution so a *single* transaction is
   scorable.
3. **XGBoost** — trained on the balanced fold with `scale_pos_weight`, early stopping on
   validation AUC-PR, and a no-SMOTE ablation to justify the choice.
4. **ROC and PR curves** across every model, plus a threshold trade-off sweep.
5. **Confusion matrices** at the operating point, with TP/FP/FN/TN spelled out in business terms.
6. **Two working UIs** — a Streamlit analyst dashboard and a Flask app with an HTML interface and
   a JSON API, both backed by the same `FraudDetector`.

**Key takeaways**

- At 0.17% prevalence, accuracy and even ROC-AUC mislead; PR-AUC against the prevalence baseline
  is the honest comparison.
- The single most damaging bug in this genre is resampling before splitting. Here the test fold
  was carved out first, and the assertion in the preprocessing cell proves the splits partition
  the dataset exactly.
- SMOTE and `scale_pos_weight` fix the same problem from different angles; the ablation shows
  which one is doing the work on this data.
- Unsupervised detectors do not replace a classifier, but they add a label-free signal that
  generalises to attack shapes the training data never contained.
- Every verdict carries its top contributing features, because a fraud model an analyst cannot
  interrogate gets ignored.